# YC Match — Fine-tune Scoring Model
Fine-tunes Qwen2.5-1.5B on 6,373 real resume-job pairs.
**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!pip install -q unsloth transformers datasets huggingface_hub trl peft

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import torch, json

print('Loading model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit',
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                     'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
)
print('Model loaded!')

In [ ]:
def format_chat(example):
    text = tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)
    return {'text': text}

train_dataset = load_dataset('json', data_files='train.jsonl', split='train')
val_dataset = load_dataset('json', data_files='val.jsonl', split='train')

train_dataset = train_dataset.map(format_chat)
val_dataset = val_dataset.map(format_chat)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        output_dir='./yc-match-scorer',
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=50,
        save_strategy='steps',
        save_steps=50,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim='adamw_8bit',
        seed=42,
        max_seq_length=2048,
        dataset_text_field='text',
        packing=True,
    ),
)

print('Training...')
stats = trainer.train()
print(f'Done! Loss: {stats.training_loss:.4f}')

In [ ]:
# Test the model
FastLanguageModel.for_inference(model)

test_messages = [
    {'role': 'system', 'content': 'You are an expert technical recruiter scoring resume-job matches. Output ONLY valid JSON with techScore, industryScore, stageScore, hiringScore (each 0-25), and explanation.'},
    {'role': 'user', 'content': 'Score this match:\n\nRESUME:\nSenior fullstack engineer, 5 years. TypeScript, React, Next.js, PostgreSQL, AWS, Docker.\n\nJOB DESCRIPTION:\nWe are looking for a senior backend engineer with experience in Go, PostgreSQL, and AWS to build payment infrastructure.'},
]

inputs = tokenizer.apply_chat_template(test_messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.1)
result = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print('Model output:')
print(result)

In [ ]:
# Push to Hugging Face
# Change this to YOUR username!
HF_USERNAME = 'YOUR_HF_USERNAME'

from huggingface_hub import login
login()  # This will prompt you for your HF token

model.push_to_hub_merged(
    f'{HF_USERNAME}/yc-match-scorer',
    tokenizer,
    save_method='merged_16bit',
)
print(f'Model pushed to https://huggingface.co/{HF_USERNAME}/yc-match-scorer')